# Ising ladder symmetries

Find several Clifford symmetries of an open transverse-field Ising ladder with the graph automorphism method, then compare them with the existing Ising heuristic symmetry.

In [1]:
import contextlib
import io
from time import perf_counter

import numpy as np

In [2]:
from sympleq.core.circuits import Gate
from sympleq.core.circuits.gate_decomposition_to_circuit import gate_to_circuit
from sympleq.core.graphs.igraph_automorphism import find_igraph_clifford_symmetries
from sympleq.core.circuits.phase_correction import clifford_phase_decomposition
from sympleq.core.symmetries.block_decomposition import block_decompose_optimal
from sympleq.core.symmetries.clifford import qudit_cost
from sympleq.models.Ising import heuristic_clifford_symmetry, ising_2d_hamiltonian

## Model

The ladder is represented as an open `n_x x n_y` square-lattice Ising model. For the usual two-leg ladder, set `n_x = 2` and vary `n_y`.

In [3]:
n_x = 2
n_y = 5
num_graph_symmetries = 8
J_zz = 1.0
h_x = 0.5
periodic = False

H = ising_2d_hamiltonian(n_x, n_y, J_zz=J_zz, h_x=h_x, periodic=periodic)

print(f"qubits: {H.n_qudits()}")
print(f"Pauli terms: {H.n_paulis()}")
print(f"LCM: {H.lcm}")

qubits: 10
Pauli terms: 23
LCM: 2


## Helpers

In [4]:
def timed(label, fn):
    start = perf_counter()
    value = fn()
    return label, value, perf_counter() - start


def quietly(fn):
    stream = io.StringIO()
    with contextlib.redirect_stdout(stream):
        return fn()


def validates_symmetry(gate, hamiltonian):
    circuit = gate_to_circuit(gate, [2] * hamiltonian.n_qudits())
    acted = circuit.act(hamiltonian).to_standard_form()
    target = hamiltonian.to_standard_form()
    return acted == target


def xz_permutation_from_gate(gate):
    F = np.asarray(gate.symplectic, dtype=int) % 2
    n = F.shape[0] // 2
    A = F[:n, :n]
    B = F[:n, n:]
    C = F[n:, :n]
    D = F[n:, n:]
    if np.array_equal(A, D) and not B.any() and not C.any() and np.all(A.sum(axis=0) == 1) and np.all(A.sum(axis=1) == 1):
        return [int(np.argmax(A[:, i])) for i in range(n)]
    return None


def same_symplectic(left, right):
    return bool(np.array_equal(left.symplectic, right.symplectic))


def dense_reversal_shear_symmetry(n_spins):
    A = np.zeros((n_spins, n_spins), dtype=int)
    B = np.ones((n_spins, n_spins), dtype=int)
    C = np.zeros((n_spins, n_spins), dtype=int)
    for i in range(n_spins):
        A[-i - 1, i] = 1
    F = np.block([[A, B], [C, A]])
    phase = np.zeros(2 * n_spins, dtype=int)
    return Gate("dense_reversal_shear", F, phase)


def gate_summary(label, gate, seconds, hamiltonian, checked=None, same_as_heuristic=None):
    permutation = xz_permutation_from_gate(gate)
    return {
        "method": label,
        "seconds": round(seconds, 6),
        "valid": validates_symmetry(gate, hamiltonian),
        "qudit_cost": qudit_cost(gate, int(hamiltonian.lcm)),
        "same_as_heuristic": same_as_heuristic,
        "checked_graph_candidates": checked,
        "qubit_permutation_if_simple": permutation,
    }

## Graph automorphism symmetry

In [5]:
label, graph_result, graph_seconds = timed(
    "igraph",
    lambda: quietly(lambda: find_igraph_clifford_symmetries(H, num_symmetries=num_graph_symmetries)),
)
graph_symmetries, checked_graph_candidates = graph_result

if not graph_symmetries:
    raise RuntimeError("The graph method did not find a non-identity Clifford symmetry.")

graph_gate = graph_symmetries[0]
print(f"found {len(graph_symmetries)} graph symmetries in {graph_seconds:.6f} s")
print(f"checked graph candidates: {checked_graph_candidates}")
for idx, gate in enumerate(graph_symmetries):
    print(f"graph symmetry {idx}: valid={validates_symmetry(gate, H)}, simple permutation={xz_permutation_from_gate(gate)}")

graph_gate.symplectic

found 3 graph symmetries in 0.536059 s
checked graph candidates: 3
graph symmetry 0: valid=True, simple permutation=[8, 9, 6, 7, 4, 5, 2, 3, 0, 1]
graph symmetry 1: valid=True, simple permutation=[1, 0, 3, 2, 5, 4, 7, 6, 9, 8]
graph symmetry 2: valid=True, simple permutation=[9, 8, 7, 6, 5, 4, 3, 2, 1, 0]


array([[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0,

## Heuristic symmetry

In [6]:
label, heuristic_gate, heuristic_seconds = timed(
    "heuristic",
    lambda: heuristic_clifford_symmetry(H.n_qudits(), periodic=periodic),
)

print(f"built heuristic in {heuristic_seconds:.6f} s")
print(f"valid symmetry: {validates_symmetry(heuristic_gate, H)}")
heuristic_gate.symplectic

built heuristic in 0.000092 s
valid symmetry: True


array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0,

## Dense cost-dropping symmetry

This symmetry has a dense standard-basis symplectic matrix, but its minimal representation after the basis transformation has much lower qubit cost.

In [7]:
label, dense_gate, dense_seconds = timed(
    "dense reversal shear",
    lambda: dense_reversal_shear_symmetry(H.n_qudits()),
)

dense_S_symplectic, dense_T_symplectic = block_decompose_optimal(
    dense_gate.symplectic,
    int(H.lcm),
    min_block_size=4,
)
dense_S_phase, dense_T_phase = clifford_phase_decomposition(
    dense_gate.symplectic,
    dense_gate.phase_vector(),
    dense_S_symplectic,
    dense_T_symplectic,
    int(H.lcm),
)
dense_minimal_gate = Gate("dense_S_min", dense_S_symplectic, dense_S_phase)

print(f"built dense symmetry in {dense_seconds:.6f} s")
print("valid symmetry: skipped here to avoid synthesizing the dense standard-basis circuit")
print(f"standard-basis qudit cost: {qudit_cost(dense_gate, int(H.lcm))}")
print(f"minimal-representation qudit cost: {qudit_cost(dense_minimal_gate, int(H.lcm))}")
dense_gate.symplectic

built dense symmetry in 0.000169 s
valid symmetry: skipped here to avoid synthesizing the dense standard-basis circuit
standard-basis qudit cost: 10
minimal-representation qudit cost: 1


array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0,

## Comparison

For this row-major open ladder, the graph method may return local geometric reflections and products of those reflections. The heuristic returns the dense anti-diagonal Ising-chain-style Clifford. This cell checks whether any graph-returned symmetry is exactly the same symplectic matrix as the heuristic.

In [8]:
summaries = [
    gate_summary(
        f"igraph[{idx}]",
        gate,
        graph_seconds,
        H,
        checked_graph_candidates,
        same_as_heuristic=same_symplectic(gate, heuristic_gate),
    )
    for idx, gate in enumerate(graph_symmetries)
]
summaries.append(gate_summary("heuristic", heuristic_gate, heuristic_seconds, H, same_as_heuristic=True))
summaries.append({
    "method": "dense reversal shear",
    "seconds": round(dense_seconds, 6),
    "valid": "not checked by circuit synthesis",
    "qudit_cost": qudit_cost(dense_gate, int(H.lcm)),
    "minimal_qudit_cost": qudit_cost(dense_minimal_gate, int(H.lcm)),
    "same_as_heuristic": same_symplectic(dense_gate, heuristic_gate),
    "checked_graph_candidates": None,
    "qubit_permutation_if_simple": xz_permutation_from_gate(dense_gate),
})

for row in summaries:
    print(row)

assert all(row["valid"] is True for row in summaries if row["method"] != "dense reversal shear")

matching_graph_indices = [
    idx for idx, gate in enumerate(graph_symmetries)
    if same_symplectic(gate, heuristic_gate)
]
print(f"graph symmetries matching heuristic: {matching_graph_indices}")

if matching_graph_indices:
    print("At least one graph symmetry matches the heuristic exactly.")
else:
    print("No graph-returned symmetry matches the heuristic exactly for this requested count.")

{'method': 'igraph[0]', 'seconds': 0.536059, 'valid': True, 'qudit_cost': 2, 'same_as_heuristic': False, 'checked_graph_candidates': 3, 'qubit_permutation_if_simple': [8, 9, 6, 7, 4, 5, 2, 3, 0, 1]}
{'method': 'igraph[1]', 'seconds': 0.536059, 'valid': True, 'qudit_cost': 2, 'same_as_heuristic': False, 'checked_graph_candidates': 3, 'qubit_permutation_if_simple': [1, 0, 3, 2, 5, 4, 7, 6, 9, 8]}
{'method': 'igraph[2]', 'seconds': 0.536059, 'valid': True, 'qudit_cost': 2, 'same_as_heuristic': True, 'checked_graph_candidates': 3, 'qubit_permutation_if_simple': [9, 8, 7, 6, 5, 4, 3, 2, 1, 0]}
{'method': 'heuristic', 'seconds': 9.2e-05, 'valid': True, 'qudit_cost': 2, 'same_as_heuristic': True, 'checked_graph_candidates': None, 'qubit_permutation_if_simple': [9, 8, 7, 6, 5, 4, 3, 2, 1, 0]}
{'method': 'dense reversal shear', 'seconds': 0.000169, 'valid': 'not checked by circuit synthesis', 'qudit_cost': 10, 'minimal_qudit_cost': 1, 'same_as_heuristic': False, 'checked_graph_candidates': None

## Size sweep

A small sweep keeps the notebook fast while checking that both methods continue to produce valid symmetries as the ladder grows.

In [9]:
rows = []
for width in range(2, 8):
    H_width = ising_2d_hamiltonian(n_x, width, J_zz=J_zz, h_x=h_x, periodic=False)

    _, graph_result, graph_t = timed(
        "igraph",
        lambda H_width=H_width: quietly(lambda: find_igraph_clifford_symmetries(H_width, num_symmetries=num_graph_symmetries)),
    )
    graph_gates, graph_checked = graph_result
    graph_valid = bool(graph_gates) and all(validates_symmetry(gate, H_width) for gate in graph_gates)

    _, heuristic_gate_width, heuristic_t = timed(
        "heuristic",
        lambda H_width=H_width: heuristic_clifford_symmetry(H_width.n_qudits(), periodic=False),
    )
    heuristic_valid = validates_symmetry(heuristic_gate_width, H_width)
    graph_matches_heuristic = [
        idx for idx, gate in enumerate(graph_gates)
        if same_symplectic(gate, heuristic_gate_width)
    ]

    rows.append({
        "n_x": n_x,
        "n_y": width,
        "qubits": H_width.n_qudits(),
        "paulis": H_width.n_paulis(),
        "igraph_s": round(graph_t, 6),
        "igraph_found": len(graph_gates),
        "igraph_checked": graph_checked,
        "igraph_valid": graph_valid,
        "igraph_matches_heuristic": graph_matches_heuristic,
        "heuristic_s": round(heuristic_t, 6),
        "heuristic_valid": heuristic_valid,
    })

for row in rows:
    print(row)

assert all(row["igraph_valid"] and row["heuristic_valid"] for row in rows)

{'n_x': 2, 'n_y': 2, 'qubits': 4, 'paulis': 8, 'igraph_s': 0.009887, 'igraph_found': 7, 'igraph_checked': 7, 'igraph_valid': True, 'igraph_matches_heuristic': [6], 'heuristic_s': 3.6e-05, 'heuristic_valid': True}
{'n_x': 2, 'n_y': 3, 'qubits': 6, 'paulis': 13, 'igraph_s': 0.008264, 'igraph_found': 3, 'igraph_checked': 3, 'igraph_valid': True, 'igraph_matches_heuristic': [2], 'heuristic_s': 3.9e-05, 'heuristic_valid': True}
{'n_x': 2, 'n_y': 4, 'qubits': 8, 'paulis': 18, 'igraph_s': 0.01177, 'igraph_found': 3, 'igraph_checked': 3, 'igraph_valid': True, 'igraph_matches_heuristic': [2], 'heuristic_s': 4.1e-05, 'heuristic_valid': True}
{'n_x': 2, 'n_y': 5, 'qubits': 10, 'paulis': 23, 'igraph_s': 0.017288, 'igraph_found': 3, 'igraph_checked': 3, 'igraph_valid': True, 'igraph_matches_heuristic': [2], 'heuristic_s': 4.6e-05, 'heuristic_valid': True}
{'n_x': 2, 'n_y': 6, 'qubits': 12, 'paulis': 28, 'igraph_s': 0.022768, 'igraph_found': 3, 'igraph_checked': 3, 'igraph_valid': True, 'igraph_matc

## Circuit for the relevant symmetry

In [11]:
relevant_label = "dense reversal shear"
relevant_gate = dense_gate

S_symplectic, T_symplectic = block_decompose_optimal(
    relevant_gate.symplectic,
    int(H.lcm),
    min_block_size=4,
)
S_phase, T_phase = clifford_phase_decomposition(
    relevant_gate.symplectic,
    relevant_gate.phase_vector(),
    S_symplectic,
    T_symplectic,
    int(H.lcm),
)
minimal_gate = Gate("S_min", S_symplectic, S_phase)
basis_transform_gate = Gate("T_basis", T_symplectic, T_phase)
symmetry_qudit_cost = qudit_cost(relevant_gate, int(H.lcm))
minimal_s_qudit_cost = qudit_cost(minimal_gate, int(H.lcm))

synthesize_basis_transform_circuit = True
minimal_circuit = gate_to_circuit(minimal_gate, [2] * H.n_qudits())
basis_transform_circuit = None
if synthesize_basis_transform_circuit:
    basis_transform_circuit = gate_to_circuit(basis_transform_gate, [2] * H.n_qudits())

print(f"Standard-basis matrix for {relevant_label} symmetry F:")
print("qudit cost of selected symmetry F:", symmetry_qudit_cost)
print(relevant_gate.symplectic)
print("phase:", relevant_gate.phase_vector())

print("\nMinimal qubit representation S:")
print("qudit cost of minimal S:", minimal_s_qudit_cost)
print(minimal_gate.symplectic)
print("phase:", minimal_gate.phase_vector())
print("\nCircuit for minimal representation S:")
print(minimal_circuit)

print("\nBasis transformation Clifford T:")
print(basis_transform_gate.symplectic)
print("phase:", basis_transform_gate.phase_vector())
if basis_transform_circuit is None:
    print("\nCircuit for basis transformation T skipped by default; set synthesize_basis_transform_circuit = True to generate it.")
else:
    print("\nCircuit for basis transformation T:")
    print(basis_transform_circuit)

Standard-basis matrix for dense reversal shear symmetry F:
qudit cost of selected symmetry F: 10
[[0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 1 0 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 1 0 0 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 1 0 0 0 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 1 0 0 0 0 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 1 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 1 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1]
 [0 0 1 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1]
 [0 1 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1]
 [1 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0]]
phase: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 